# Notebook 3: DETR Fine-tuning on BDD100K
**Author:** Mahmoud Abdulkareem

> **Prerequisite:** Run `01_dataset_preparation.ipynb` once before this.

## Step 1: Mount Drive + Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers torchvision pycocotools

In [ ]:
import json, time, random, zipfile, shutil
from pathlib import Path
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import DetrForObjectDetection, DetrImageProcessor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Step 3: Config

In [ ]:
DRIVE_ROOT    = Path('/content/drive/MyDrive/vehicle_detection')
DRIVE_BACKUP  = DRIVE_ROOT / 'processed'
DRIVE_RESULTS = DRIVE_ROOT / 'results' / 'detr'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

LOCAL_BDD  = Path('/content/bdd100k')
LOCAL_DATA = Path('/content/data')

MODEL_CKPT      = 'facebook/detr-resnet-50'
VEHICLE_CLASSES = ['car', 'truck', 'bus', 'motorcycle']
EPOCHS          = 50
BATCH_SIZE      = 4      # reduce to 2 if OOM
LR              = 1e-4
LR_BACKBONE     = 1e-5
WEIGHT_DECAY    = 1e-4
IMG_SIZE        = 800
PATIENCE        = 10

assert DRIVE_BACKUP.exists(), 'Run Notebook 1 first.'
print('Config OK')

## Step 4: Session Setup
Unzips images locally and restores COCO JSONs from Drive. Run every new Colab session.

In [ ]:
DRIVE_ZIP = DRIVE_ROOT / 'downloads' / 'solesensei_bdd100k.zip'

if not LOCAL_BDD.exists():
    print('Unzipping BDD100K to local disk (~5 min)...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall('/content/')
    print('Unzip done.')

if not (LOCAL_BDD / 'images').exists() and (LOCAL_BDD / 'bdd100k' / 'images').exists():
    LOCAL_BDD = LOCAL_BDD / 'bdd100k'

if not LOCAL_DATA.exists():
    print('Restoring label files from Drive...')
    shutil.copytree(DRIVE_BACKUP, LOCAL_DATA)
    print('Done.')

COCO_ANN = LOCAL_DATA / 'coco' / 'annotations'
assert COCO_ANN.exists(), f'COCO annotations not found: {COCO_ANN}'
print('Session setup complete.')
print('  Images   :', LOCAL_BDD / 'images' / '100k')
print('  COCO ann :', COCO_ANN)

## Step 5: Dataset

In [ ]:
class BDDCocoDataset(Dataset):
    def __init__(self, bdd_root, ann_json, processor, augment=False):
        self.bdd_root  = Path(bdd_root)
        self.coco      = COCO(ann_json)
        self.ids       = list(self.coco.imgs.keys())
        self.processor = processor
        self.augment   = augment

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id   = self.ids[idx]
        img_info = self.coco.imgs[img_id]
        # file_name is stored as relative path from bdd_root
        img_path = self.bdd_root / img_info['file_name']
        image    = Image.open(img_path).convert('RGB')
        W, H     = image.size

        anns   = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        boxes  = [a['bbox'] for a in anns]
        labels = [a['category_id'] for a in anns]

        do_flip = self.augment and random.random() > 0.5
        if do_flip:
            image = F.hflip(image)
            boxes = [[W - x - w, y, w, h] for x, y, w, h in boxes]
        if self.augment:
            image = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)(image)

        boxes_xyxy = [[x, y, x+w, y+h] for x, y, w, h in boxes]
        target = {
            'image_id': torch.tensor([img_id]),
            'annotations': [{'bbox': b, 'category_id': l,
                             'area': (b[2]-b[0])*(b[3]-b[1]), 'iscrowd': 0}
                            for b, l in zip(boxes_xyxy, labels)],
        }
        enc = self.processor(images=image, annotations=target, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}


def collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'pixel_mask':   torch.stack([b['pixel_mask']   for b in batch]),
        'labels':       [b['labels']                   for b in batch],
    }

In [ ]:
processor = DetrImageProcessor.from_pretrained(MODEL_CKPT, size={'shortest_edge': IMG_SIZE})

train_ds = BDDCocoDataset(LOCAL_BDD, COCO_ANN / 'instances_train.json', processor, augment=True)
val_ds   = BDDCocoDataset(LOCAL_BDD, COCO_ANN / 'instances_val.json',   processor, augment=False)
test_ds  = BDDCocoDataset(LOCAL_BDD, COCO_ANN / 'instances_test.json',  processor, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'Train : {len(train_ds):,}')
print(f'Val   : {len(val_ds):,}')
print(f'Test  : {len(test_ds):,}')

## Step 6: Load DETR Model

In [ ]:
with open(COCO_ANN / 'instances_train.json') as f:
    coco_meta = json.load(f)
id2label = {cat['id']: cat['name'] for cat in coco_meta['categories']}
label2id = {v: k for k, v in id2label.items()}
print('id2label:', id2label)

model = DetrForObjectDetection.from_pretrained(
    MODEL_CKPT, num_labels=len(id2label),
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
).to(DEVICE)

total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable:,}')

## Step 7: Training Loop

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'backbone' in n], 'lr': LR_BACKBONE},
    {'params': [p for n, p in model.named_parameters() if 'backbone' not in n], 'lr': LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history       = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_cnt  = 0
train_start   = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t_ep = time.time()
    for batch in train_loader:
        pv  = batch['pixel_values'].to(DEVICE)
        pm  = batch['pixel_mask'].to(DEVICE)
        lbs = [{k: v.to(DEVICE) for k, v in l.items()} for l in batch['labels']]
        optimizer.zero_grad()
        out = model(pixel_values=pv, pixel_mask=pm, labels=lbs)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        epoch_loss += out.loss.item()
    avg_train = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_train)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            pv  = batch['pixel_values'].to(DEVICE)
            pm  = batch['pixel_mask'].to(DEVICE)
            lbs = [{k: v.to(DEVICE) for k, v in l.items()} for l in batch['labels']]
            val_loss += model(pixel_values=pv, pixel_mask=pm, labels=lbs).loss.item()
    avg_val = val_loss / len(val_loader)
    history['val_loss'].append(avg_val)
    scheduler.step()

    print(f'Epoch {epoch:3d}/{EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}  time={time.time()-t_ep:.1f}s')

    if avg_val < best_val_loss:
        best_val_loss = avg_val; patience_cnt = 0
        model.save_pretrained(DRIVE_RESULTS / 'best_model')
        processor.save_pretrained(DRIVE_RESULTS / 'best_model')
        print(f'  Saved best model (val={best_val_loss:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break

total_train_time = time.time() - train_start
actual_epochs    = len(history['train_loss'])
print(f'Total time     : {total_train_time/60:.1f} min')
print(f'Time per epoch : {total_train_time/actual_epochs:.1f} s')

timing = {'model': 'DETR-ResNet50', 'total_train_time_s': round(total_train_time, 2),
          'time_per_epoch_s': round(total_train_time/actual_epochs, 2), 'epochs_trained': actual_epochs}
(DRIVE_RESULTS / 'timing.json').write_text(json.dumps(timing, indent=2))

## Step 8: Training Curves

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('DETR Training & Validation Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(str(DRIVE_RESULTS / 'detr_loss_curves.png'), dpi=120)
plt.show()

## Step 9: Evaluate on Test Set

In [ ]:
model_best = DetrForObjectDetection.from_pretrained(DRIVE_RESULTS / 'best_model').to(DEVICE).eval()
proc_best  = DetrImageProcessor.from_pretrained(DRIVE_RESULTS / 'best_model')

coco_gt    = COCO(str(COCO_ANN / 'instances_test.json'))
coco_preds = []
CONF_THRESHOLD = 0.5

with torch.no_grad():
    for batch in test_loader:
        pv  = batch['pixel_values'].to(DEVICE)
        pm  = batch['pixel_mask'].to(DEVICE)
        out = model_best(pixel_values=pv, pixel_mask=pm)
        sizes = [(l['orig_size'][0].item(), l['orig_size'][1].item()) for l in batch['labels']]
        results = proc_best.post_process_object_detection(out, threshold=CONF_THRESHOLD, target_sizes=sizes)
        for res, lbl in zip(results, batch['labels']):
            img_id = lbl['image_id'].item()
            for score, label, box in zip(res['scores'], res['labels'], res['boxes']):
                x1, y1, x2, y2 = box.tolist()
                coco_preds.append({'image_id': img_id, 'category_id': label.item(),
                                   'bbox': [x1, y1, x2-x1, y2-y1], 'score': round(score.item(), 4)})

print(f'Total predictions: {len(coco_preds):,}')

coco_dt   = coco_gt.loadRes(coco_preds)
coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()
map50, map5095 = coco_eval.stats[1], coco_eval.stats[0]

In [ ]:
gt_by_img   = defaultdict(list)
pred_by_img = defaultdict(list)
for ann in coco_gt.dataset['annotations']: gt_by_img[ann['image_id']].append(ann)
for p in coco_preds: pred_by_img[p['image_id']].append(p)

def iou_box(b1, b2):
    ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
    ix2 = min(b1[0]+b1[2], b2[0]+b2[2]); iy2 = min(b1[1]+b1[3], b2[1]+b2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
    return inter / union if union > 0 else 0

TP = FP = FN = 0
for img_id in coco_gt.imgs:
    gts = gt_by_img[img_id]; matched = [False] * len(gts)
    for pred in sorted(pred_by_img[img_id], key=lambda x: -x['score']):
        best_iou, best_j = 0, -1
        for j, gt in enumerate(gts):
            if matched[j] or gt['category_id'] != pred['category_id']: continue
            iou = iou_box(pred['bbox'], gt['bbox'])
            if iou > best_iou: best_iou, best_j = iou, j
        if best_iou >= 0.5: TP += 1; matched[best_j] = True
        else: FP += 1
    FN += matched.count(False)

prec = TP / (TP + FP + 1e-9)
rec  = TP / (TP + FN + 1e-9)
f1   = 2 * prec * rec / (prec + rec + 1e-9)
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')

## Step 10: Inference Speed

In [ ]:
test_img_paths = [
    LOCAL_BDD / img['file_name']
    for img in coco_gt.dataset['images'][:100]
]

img0 = Image.open(test_img_paths[0]).convert('RGB')
with torch.no_grad():
    _ = model_best(**proc_best(images=img0, return_tensors='pt').to(DEVICE))  # warmup

t0 = time.time()
with torch.no_grad():
    for p in test_img_paths:
        img = Image.open(p).convert('RGB')
        _ = model_best(**proc_best(images=img, return_tensors='pt').to(DEVICE))
elapsed = time.time() - t0

ms_per_img = elapsed / len(test_img_paths) * 1000
fps        = 1000 / ms_per_img
print(f'Inference : {ms_per_img:.1f} ms/image  ({fps:.1f} FPS)')

metrics_out = {
    'model': 'DETR-ResNet50',
    'map50': round(float(map50), 4), 'map50_95': round(float(map5095), 4),
    'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4),
    'inference_ms': round(ms_per_img, 2), 'fps': round(fps, 1),
    'trainable_params': trainable, 'time_per_epoch_s': timing['time_per_epoch_s'],
}
(DRIVE_RESULTS / 'metrics.json').write_text(json.dumps(metrics_out, indent=2))
print('Saved metrics.json')
print(json.dumps(metrics_out, indent=2))

## Step 11: Sample Detections

In [ ]:
COLORS = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']
sample = random.sample(test_img_paths, min(6, len(test_img_paths)))
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('DETR-ResNet50 — Sample Detections on Test Set', fontsize=13)

with torch.no_grad():
    for ax, img_path in zip(axes.flat, sample):
        img = Image.open(img_path).convert('RGB')
        enc = proc_best(images=img, return_tensors='pt').to(DEVICE)
        res = proc_best.post_process_object_detection(
            model_best(**enc), threshold=CONF_THRESHOLD, target_sizes=[img.size[::-1]]
        )[0]
        ax.imshow(img)
        for score, label, box in zip(res['scores'], res['labels'], res['boxes']):
            x1, y1, x2, y2 = box.tolist()
            idx  = label.item() % len(COLORS)
            name = id2label.get(label.item(), str(label.item()))
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                           lw=2, ec=COLORS[idx], fc='none'))
            ax.text(x1, y1-4, f'{name} {score:.2f}',
                    color=COLORS[idx], fontsize=7, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.savefig(str(DRIVE_RESULTS / 'detr_sample_detections.png'), dpi=120)
plt.show()
print('DETR training complete.')